# Mineral-map component-weight optimization

This notebook is a separate, reproducible workflow for finding non-reductive additive component weights. It keeps the scientific distance model and the exploratory force-directed layout distinct: candidate weights are evaluated against normalized component behaviour, graph stability, component diversity, and soft classification continuity before one deterministic candidate is exported.

The source dataset is the existing 6,228-mineral IMA collection. Geographic-map concepts from the generic project outline are adapted here to the mineral-composition graph: coordinates are 3D layout coordinates, regions are compositional communities, and map features are browser node records.


In [ ]:
CANDIDATE_COUNT = 512
PAIR_SAMPLE_SIZE = 100_000
STABILITY_REPLICATES = 4
GRAPH_NEIGHBORS = 10

Loaded 6,228 minerals and 62,280 directed k-NN edges.
Input metadata coordinate system: normalized 3D force-directed layout of the exact additive k-nearest-neighbour graph; edge distances remain authoritative
Robust component normalization completed.
Accepted 512 of 513 candidate weight sets.
Selected weights: {'anion_group': 0.3102713920901254, 'cations': 0.3223567163838267, 'extra_anions': 0.15809244174817924, 'hydration': 0.0906987675275306, 'structural_water': 0.07976691488131664, 'structure': 0.038813767369021236}
Effective contributions: {'anion_group': 0.1330986148101503, 'cations': 0.3940481865907539, 'extra_anions': 0.3022050863724447, 'hydration': 0.1090715470789679, 'structural_water': 0.028825596496422207, 'structure': 0.032750968651156405}
Core tests passed.
End-to-end check passed: quartz filter returned 1 minerals.
Selected graph: 6,228 nodes, 43,198 undirected edges.
Wrote c:\My Things\HTML\Demonin's Item Shop\other\minerals\component_weight_optimization_report.json and

In [5]:
nearest_neighbors = NearestNeighbors(
    n_neighbors=min(node_count, GRAPH_NEIGHBORS * 8 + 1),
    metric="euclidean",
    algorithm="auto",
    n_jobs=-1,
)
nearest_neighbors.fit(weighted_landmark_signatures)
approximate_distances, approximate_indices = nearest_neighbors.kneighbors(weighted_landmark_signatures)

# Build a fresh directed graph from the new full 6,228-mineral search. Query
# extra candidates because duplicate landmark signatures can occupy slots.
approximate_edges = []
for source in range(node_count):
    rank = 0
    seen_targets = {source}
    for column in range(1, approximate_indices.shape[1]):
        target = int(approximate_indices[source, column])
        if target in seen_targets:
            continue
        seen_targets.add(target)
        rank += 1
        approximate_edges.append({
            "source_index": source,
            "target_index": target,
            "neighbor_rank": rank,
            "distance": float(approximate_distances[source, column]),
        })
        if rank == GRAPH_NEIGHBORS:
            break
    if rank != GRAPH_NEIGHBORS:
        raise RuntimeError(f"Could not find {GRAPH_NEIGHBORS} unique neighbours for source {source}; found {rank}.")

if len(approximate_edges) != node_count * GRAPH_NEIGHBORS:
    raise RuntimeError(f"Expected {node_count * GRAPH_NEIGHBORS:,} approximate edges, got {len(approximate_edges):,}.")


In [3]:
with np.load(ROOT / "additive_component_distances_256.npz", allow_pickle=False) as cache:
    print("256-landmark cache keys:", cache.files)
    print("256-landmark cache shapes:", {key: cache[key].shape for key in cache.files})
print("component names:", COMPONENT_NAMES)


256-landmark cache keys: ['anion_group', 'cations', 'extra_anions', 'hydration', 'structure']
256-landmark cache shapes: {'anion_group': (6228, 256), 'cations': (6228, 256), 'extra_anions': (6228, 256), 'hydration': (6228, 256), 'structure': (6228, 256)}
component names: ('anion_group', 'cations', 'extra_anions', 'hydration', 'structural_water', 'structure')


In [7]:
# Continue the 256-landmark regeneration: layout, export, and diagnostics.
force_graph = nx.Graph()
force_graph.add_nodes_from(range(node_count))
edge_distance_values = np.asarray([edge["distance"] for edge in approximate_edges], dtype=np.float64)
edge_low, edge_high = np.percentile(edge_distance_values, [5, 95])
edge_lookup = {}
for edge in approximate_edges:
    pair = tuple(sorted((int(edge["source_index"]), int(edge["target_index"]))))
    similarity = 1.0 - np.clip((edge["distance"] - edge_low) / max(edge_high - edge_low, 1e-9), 0.0, 1.0)
    edge_lookup[pair] = max(float(0.2 + 0.8 * similarity), edge_lookup.get(pair, 0.0))
for (source, target), weight in edge_lookup.items():
    force_graph.add_edge(source, target, weight=weight)

fresh_initial = rng.normal(0.0, 0.35, size=(node_count, 3))
fresh_initial -= fresh_initial.mean(axis=0, keepdims=True)
fresh_initial /= max(np.max(np.linalg.norm(fresh_initial, axis=1)), 1e-9)
force_positions = nx.spring_layout(
    force_graph, dim=3, seed=SEED, iterations=FORCE_ITERATIONS, k=FORCE_K,
    scale=1.0, weight="weight", pos={index: fresh_initial[index] for index in range(node_count)},
)
regenerated_coordinates = np.asarray([force_positions[index] for index in range(node_count)], dtype=np.float64)
regenerated_coordinates -= regenerated_coordinates.mean(axis=0, keepdims=True)
regenerated_coordinates /= max(np.max(np.abs(regenerated_coordinates)), 1e-9)
regenerated_coordinates *= 0.95

with WEB_NODES.open(encoding="utf-8") as input_file:
    regenerated_nodes_payload = json.load(input_file)
with WEB_METADATA.open(encoding="utf-8") as input_file:
    regenerated_metadata = json.load(input_file)
for node, coordinates in zip(regenerated_nodes_payload["nodes"], regenerated_coordinates):
    node["coordinates"] = [float(value) for value in coordinates]
regenerated_nodes_payload["coordinateSystem"] = "normalized 3D force-directed layout from regenerated 256-landmark weighted k-NN signatures"
regenerated_metadata["componentWeights"] = dict(zip(COMPONENT_NAMES, map(float, selected_weights)))
regenerated_metadata["distanceModel"]["mapAlgorithm"] = "weighted 3D force-directed k-NN graph with 256-landmark signatures"
regenerated_metadata["distanceModel"]["mapParameters"] = {
    "dimensions": 3, "landmarkCount": LANDMARK_COUNT, "neighborsPerMineral": GRAPH_NEIGHBORS,
    "iterations": FORCE_ITERATIONS, "k": FORCE_K, "randomState": SEED,
    "undirectedEdges": force_graph.number_of_edges(), "weightSelectionReport": REPORT_PATH.name,
}
regenerated_metadata["distanceModel"]["approximation"] = {
    "method": "Euclidean nearest-neighbour search on weighted component-to-landmark signatures",
    "missingCachedComponents": sorted(missing_components),
    "exactPairwiseDistancesComputed": False,
    "warning": "Exploratory 256-landmark topology; structural_water contributes zero because it is absent from the legacy cache.",
}

approximate_neighbors_by_source = [[] for _ in range(node_count)]
for edge in approximate_edges:
    approximate_neighbors_by_source[int(edge["source_index"])].append({
        "targetId": int(edge["target_index"]), "rank": int(edge["neighbor_rank"]),
        "distance": float(edge["distance"]), "category": "approximate_landmark_neighbour",
        "components": {component: {"raw": 0.0, "weighted": 0.0} for component in COMPONENT_NAMES},
    })
regenerated_neighbor_payload = {
    "schemaVersion": "mineral-map-static-v1", "neighborCount": GRAPH_NEIGHBORS,
    "approximate": True, "neighborsBySourceId": approximate_neighbors_by_source,
}
with APPROX_NEIGHBOR_PATH.open("w", newline="", encoding="utf-8") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=["source_index", "target_index", "neighbor_rank", "distance"])
    writer.writeheader(); writer.writerows(approximate_edges)
for path, payload in (
    (WEB_NODES, regenerated_nodes_payload), (WEB_METADATA, regenerated_metadata),
    (FRONTEND_NODES, regenerated_nodes_payload), (FRONTEND_METADATA, regenerated_metadata),
    (WEB_NEIGHBORS, regenerated_neighbor_payload), (FRONTEND_NEIGHBORS, regenerated_neighbor_payload),
):
    path.write_text(json.dumps(payload, ensure_ascii=True, separators=(",", ":")) + "\n", encoding="utf-8")
print(f"Regenerated {len(approximate_edges):,} directed edges; {force_graph.number_of_edges():,} undirected edges.")


Regenerated 62,280 directed edges; 42,366 undirected edges.


In [ ]:
# 15. Reapply the original relationship taxonomy to the exported 10-NN neighbours
# Self-contained: it needs only the two exported neighbour payloads and the
# authoritative edge CSV, so it runs even after a kernel restart and touches
# ONLY the 62,280 k-NN edges the frontend displays -- never all-pairs distances.
import csv
import json
from collections import Counter

APPROXIMATE_CATEGORY = "approximate_landmark_neighbour"
APPROXIMATE_DESCRIPTION = (
    "Nearest neighbour found by the exploratory 256-landmark signature search; "
    "relationship class is pending exact pairwise recomputation."
)

with WEB_METADATA.open(encoding="utf-8") as input_file:
    metadata_payload = json.load(input_file)
taxonomy_descriptions = dict(metadata_payload.get("relationshipCategories", {}))
taxonomy_descriptions.setdefault(APPROXIMATE_CATEGORY, APPROXIMATE_DESCRIPTION)

# One linear scan of the 13 MB edge CSV (about a second) to recover the taxonomy
# label that was stored with every original directed edge.
old_category_by_pair = {}
with EDGE_CSV.open(newline="", encoding="utf-8-sig") as input_file:
    for row in csv.DictReader(input_file):
        category = row.get("relationship_category", "")
        if category:
            old_category_by_pair[(int(row["source_index"]), int(row["target_index"]))] = category

with WEB_NEIGHBORS.open(encoding="utf-8") as input_file:
    neighbor_payload = json.load(input_file)

category_counts = Counter()
for source_id, source_neighbors in enumerate(neighbor_payload["neighborsBySourceId"]):
    for entry in source_neighbors:
        old_category = old_category_by_pair.get((source_id, int(entry["targetId"])))
        entry["category"] = old_category or APPROXIMATE_CATEGORY
        category_counts[entry["category"]] += 1

regenerated_metadata["relationshipCategories"] = taxonomy_descriptions
for path, payload in (
    (WEB_METADATA, regenerated_metadata),
    (FRONTEND_METADATA, regenerated_metadata),
    (WEB_NEIGHBORS, neighbor_payload),
    (FRONTEND_NEIGHBORS, neighbor_payload),
):
    path.write_text(json.dumps(payload, ensure_ascii=True, separators=(",", ":")) + "\n", encoding="utf-8")

print(f"Relabelled {sum(category_counts.values()):,} k-NN edges:")
for category, count in category_counts.most_common():
    print(f"  {category}: {count:,}")


In [8]:
# 14. Quantify whether the regenerated topology actually changed
old_by_source = [[] for _ in range(node_count)]
with EDGE_CSV.open(newline="", encoding="utf-8-sig") as input_file:
    for row in csv.DictReader(input_file):
        old_by_source[int(row["source_index"])].append(int(row["target_index"]))
new_by_source = [[] for _ in range(node_count)]
with APPROX_NEIGHBOR_PATH.open(newline="", encoding="utf-8-sig") as input_file:
    for row in csv.DictReader(input_file):
        new_by_source[int(row["source_index"])].append(int(row["target_index"]))

per_source_overlap = np.asarray([
    len(set(old_targets) & set(new_targets)) / max(len(set(old_targets) | set(new_targets)), 1)
    for old_targets, new_targets in zip(old_by_source, new_by_source)
])
per_source_recall = np.asarray([
    len(set(old_targets) & set(new_targets)) / max(len(set(old_targets)), 1)
    for old_targets, new_targets in zip(old_by_source, new_by_source)
])
old_pairs = {tuple(sorted((source, target))) for source, targets in enumerate(old_by_source) for target in targets}
new_pairs = {tuple(sorted((source, target))) for source, targets in enumerate(new_by_source) for target in targets}
print(f"Directed neighbour-set Jaccard: {per_source_overlap.mean():.4f}")
print(f"Directed old-neighbour recall: {per_source_recall.mean():.4f}")
print(f"Directed edges retained: {sum(len(set(old) & set(new)) for old, new in zip(old_by_source, new_by_source)):,} / {node_count * GRAPH_NEIGHBORS:,}")
print(f"Undirected edge-set Jaccard: {len(old_pairs & new_pairs) / max(len(old_pairs | new_pairs), 1):.4f}")
print(f"Sources with at least one changed neighbour: {sum(set(old) != set(new) for old, new in zip(old_by_source, new_by_source)):,} / {node_count:,}")


Directed neighbour-set Jaccard: 0.2784
Directed old-neighbour recall: 0.3939
Directed edges retained: 24,535 / 62,280
Undirected edge-set Jaccard: 0.2500
Sources with at least one changed neighbour: 6,205 / 6,228
